In [ ]:
# Run this cell first to install required packages (Colab only)
!pip install "numpy<2" roboticstoolbox-python spatialmath-python -q

# Restart runtime so the new numpy is picked up (required on Colab)
import os, time
if 'COLAB_RELEASE_TAG' in os.environ:
    print('\n⏳ Restarting runtime to load numpy<2 — please re-run all cells after restart.')
    time.sleep(1)
    os.kill(os.getpid(), 9)

# Lab 2: DH Forward Kinematics
## EE0849 Introduction to Robotics

In this lab you will:
1. Build DH robot models with `roboticstoolbox-python`
2. Compute forward kinematics and verify by hand
3. Visualize robot configurations

---
## DH Convention Review

| Parameter | Symbol | Description |
|-----------|--------|-------------|
| Joint angle | $\theta_i$ | Angle about $z_{i-1}$ from $x_{i-1}$ to $x_i$ |
| Link offset | $d_i$ | Distance along $z_{i-1}$ from $x_{i-1}$ to $x_i$ |
| Link length | $a_i$ | Distance along $x_i$ from $z_{i-1}$ to $z_i$ |
| Link twist | $\alpha_i$ | Angle about $x_i$ from $z_{i-1}$ to $z_i$ |

The DH transformation matrix:

$$
T_i^{i-1} = \begin{bmatrix}
c\theta_i & -s\theta_i \, c\alpha_i & s\theta_i \, s\alpha_i & a_i \, c\theta_i \\
s\theta_i & c\theta_i \, c\alpha_i & -c\theta_i \, s\alpha_i & a_i \, s\theta_i \\
0 & s\alpha_i & c\alpha_i & d_i \\
0 & 0 & 0 & 1
\end{bmatrix}
$$

---
## Part 1: 2-Link Planar Arm

- Link 1: $L_1 = 1.0$ m, Link 2: $L_2 = 0.5$ m
- Both revolute, all $\alpha_i = 0$, all $d_i = 0$

| Joint | $\theta_i$ | $d_i$ | $a_i$ | $\alpha_i$ |
|-------|-----------|-------|-------|-----------|
| 1     | $\theta_1$ | 0     | 1.0   | 0         |
| 2     | $\theta_2$ | 0     | 0.5   | 0         |

In [ ]:
import numpy as np
from roboticstoolbox import DHRobot, RevoluteDH

L1, L2 = 1.0, 0.5

planar_2link = DHRobot([
    RevoluteDH(d=0, a=L1, alpha=0),
    RevoluteDH(d=0, a=L2, alpha=0),
], name='Planar-2R')

print(planar_2link)

### Exercise 1: Verify your hand calculation

Compute FK for $q_1 = 30°$, $q_2 = 45°$. Compare with:

$x = L_1 \cos(30°) + L_2 \cos(75°)$, $\quad y = L_1 \sin(30°) + L_2 \sin(75°)$

In [ ]:
# YOUR CODE HERE
q = [0, 0]  # replace with correct angles in radians
T = planar_2link.fkine(q)
print("End-effector position:", T.t)

### Visualizing the 2-Link Arm

In [ ]:
import matplotlib.pyplot as plt

def plot_planar_arm(robot, q, title=""):
    """Simple stick-figure plot of a planar arm."""
    # Compute joint positions
    points = [[0, 0]]
    T = np.eye(4)
    for i, link in enumerate(robot.links):
        T = T @ link.A(q[i]).A
        points.append([T[0, 3], T[1, 3]])
    points = np.array(points)

    plt.figure(figsize=(6, 6))
    plt.plot(points[:, 0], points[:, 1], 'o-', lw=4, markersize=10, color='steelblue')
    plt.plot(points[-1, 0], points[-1, 1], 's', markersize=12, color='orange', label='End-effector')
    plt.plot(0, 0, 'ko', markersize=12)  # base
    plt.xlabel('x (m)')
    plt.ylabel('y (m)')
    plt.title(title)
    plt.axis('equal')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

# Plot a few configurations
plot_planar_arm(planar_2link, [0, 0], "Zero configuration")
plot_planar_arm(planar_2link, [np.pi/4, np.pi/4], "θ₁=45°, θ₂=45°")
plot_planar_arm(planar_2link, [np.pi/2, -np.pi/4], "θ₁=90°, θ₂=-45°")

---
## Part 2: FANUC ER-4iA (6-DOF)

| Joint | $\theta_i$ | $d_i$ (mm) | $a_i$ (mm) | $\alpha_i$ |
|-------|-----------|-----------|-----------|----------|
| 1     | $\theta_1$ | 330       | 50        | $-90°$   |
| 2     | $\theta_2$ | 0         | 330       | $0°$     |
| 3     | $\theta_3$ | 0         | 35        | $-90°$   |
| 4     | $\theta_4$ | 335       | 0         | $90°$    |
| 5     | $\theta_5$ | 0         | 0         | $-90°$   |
| 6     | $\theta_6$ | 80        | 0         | $0°$     |

In [ ]:
from spatialmath import SE3

# FANUC ER-4iA dimensions (meters)
fanuc = DHRobot([
    RevoluteDH(d=0.330, a=0.050, alpha=-np.pi/2),
    RevoluteDH(d=0,     a=0.330, alpha=0),
    RevoluteDH(d=0,     a=0.035, alpha=-np.pi/2),
    RevoluteDH(d=0.335, a=0,     alpha=np.pi/2),
    RevoluteDH(d=0,     a=0,     alpha=-np.pi/2),
    RevoluteDH(d=0.080, a=0,     alpha=0),
], name='FANUC ER-4iA')

print(fanuc)

In [ ]:
# Test configurations
configs = {
    'zero':        [0, 0, 0, 0, 0, 0],
    'shoulder up': [0, -np.pi/4, 0, 0, 0, 0],
    'elbow bent':  [0, 0, np.pi/2, 0, 0, 0],
    'waist turn':  [np.pi/2, 0, 0, 0, 0, 0],
}

for name, q in configs.items():
    pos = fanuc.fkine(q).t * 1000  # mm
    print(f"{name:15s} -> x={pos[0]:.1f}, y={pos[1]:.1f}, z={pos[2]:.1f} mm")

### Visualizing the FANUC

In [ ]:
def get_joint_positions(robot, q):
    """Compute world position of each joint."""
    points = [[0, 0, 0]]
    T = robot.base
    for i in range(robot.n):
        T = T * robot.links[i].A(q[i])
        points.append(T.t.tolist())
    return np.array(points)

def plot_robot(robot, q, title=""):
    """Side view (XZ) and top view (XY) of a robot."""
    pts = get_joint_positions(robot, q)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    for ax, c1, c2, lbl1, lbl2, name in [
        (ax1, 0, 2, 'X (m)', 'Z (m)', 'Side view (XZ)'),
        (ax2, 0, 1, 'X (m)', 'Y (m)', 'Top view (XY)'),
    ]:
        ax.plot(pts[:, c1], pts[:, c2], 'o-', lw=3, markersize=8, color='steelblue')
        ax.plot(pts[-1, c1], pts[-1, c2], 'D', markersize=10, color='orange', label='End-effector')
        ax.plot(pts[0, c1], pts[0, c2], 'ko', markersize=10, label='Base')
        ax.set_xlabel(lbl1)
        ax.set_ylabel(lbl2)
        ax.set_title(name)
        ax.axis('equal')
        ax.grid(True, alpha=0.3)
        ax.legend()

    fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_robot(fanuc, [0, 0, 0, 0, 0, 0], "Zero configuration")
plot_robot(fanuc, [0, -np.pi/4, np.pi/4, 0, 0, 0], "Shoulder + elbow")

### Exercise 2: Calculate zero-position by hand

At $q = [0,0,0,0,0,0]$, trace through each DH transform and predict the end-effector position. Compare with `fkine()`.

In [ ]:
# YOUR CODE HERE
# Compute fkine at zero config and print position
T_zero = fanuc.fkine([0, 0, 0, 0, 0, 0])
print("Position (mm):", T_zero.t * 1000)
print("Rotation:\n", np.round(T_zero.R, 3))

### Exercise 3: Find a target pose

Use trial and error to find joint angles that place the end-effector near $(0.5, 0, 0.4)$ m.

*Hint: Start by adjusting $\theta_2$ (shoulder) to lift the arm.*

In [ ]:
# YOUR CODE HERE — adjust these angles
q_guess = [0, 0, 0, 0, 0, 0]

T = fanuc.fkine(q_guess)
target = np.array([0.5, 0, 0.4])
error = np.linalg.norm(T.t - target) * 1000

print("Achieved:", np.round(T.t, 3))
print(f"Error: {error:.1f} mm from target")

plot_robot(fanuc, q_guess, f"Error: {error:.0f} mm")